# Nested grouped development comparison
Run notebook 01 first. This compares tuning procedures on development recordings, not independent final performance. Runtime depends on your CPU. All searches use one worker to avoid memory pressure.

In [1]:
from pathlib import Path
import json, hashlib, platform
import numpy as np
import pandas as pd
import sklearn
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p/'research/src/supervised/final_holdout.py').exists())
BASE=ROOT/'data/processed/supervised/v1'
ART=BASE/'assignment_artifacts_20261030T020845905141Z'
OUT=ROOT/'data/processed/supervised/development_extension_v2'
OUT.mkdir(parents=True,exist_ok=True)
features=json.loads((ART/'artifact_manifest.json').read_text())['feature_allowlist']
df=pd.read_csv(ART/'development_train.csv',dtype={'recording_id':str})
assert len(df)==184 and df.recording_id.is_unique
assert len(features)==50 and np.isfinite(df[features].to_numpy()).all()
assert not set(features)&{'model_state','location_group','recording_id','scientific_name','latitude','longitude'}
X=df[features]; y=df.model_state; groups=df.location_group
CLASSES=['AZ','CA','TX']
assert set(y)==set(CLASSES) and groups.notna().all()
print('Development only:',len(df),'recordings;',groups.nunique(),'coordinate groups')

Development only: 184 recordings; 136 coordinate groups


In [2]:
import sys
sys.path.insert(0,str(ROOT/'research/src'))
from supervised.representations import make_representations
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import StratifiedGroupKFold, GridSearchCV
from sklearn.metrics import accuracy_score, balanced_accuracy_score, f1_score, confusion_matrix, make_scorer
from sklearn.naive_bayes import MultinomialNB, BernoulliNB, GaussianNB
from sklearn.tree import DecisionTreeClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.dummy import DummyClassifier
protocol=json.loads((OUT/'protocol.json').read_text())
assert protocol['development_sha256']==hashlib.sha256((ART/'development_train.csv').read_bytes()).hexdigest()
folds=pd.read_csv(OUT/'outer_folds.csv',dtype={'recording_id':str}).set_index('recording_id').loc[df.recording_id].outer_fold.to_numpy()
assert set(folds)==set(range(5))
def pipe(rep,model): return Pipeline([('representation',rep),('model',model)])
reps=make_representations()
models={
 'Prior':(DummyClassifier(strategy='prior'),{}),
 'Multinomial NB':(pipe(reps['multinomial_bins'],MultinomialNB()),{'representation__encode__n_bins':[3,5,8],'model__alpha':[0.5,1,2]}),
 'Bernoulli NB':(pipe(reps['bernoulli_indicators'],BernoulliNB(binarize=None)),{'model__alpha':[0.5,1,2]}),
 'Gaussian NB':(pipe(reps['continuous'],GaussianNB()),{'model__var_smoothing':[1e-9,1e-7,1e-5]}),
 'Decision tree':(pipe(reps['continuous'],DecisionTreeClassifier(random_state=5612)),{'model__criterion':['gini','entropy'],'model__max_depth':[2,3,5],'model__min_samples_leaf':[5,10]}),
 'Logistic regression':(Pipeline([('scale',StandardScaler()),('model',LogisticRegression(max_iter=5000,random_state=5612))]),{'model__C':[0.01,0.1,1,10]}),
 'Random forest':(RandomForestClassifier(n_estimators=150,random_state=5612,n_jobs=1),{'max_depth':[3,None],'min_samples_leaf':[5,10],'max_features':['sqrt',0.5]})}
scorer=make_scorer(f1_score,labels=CLASSES,average='macro',zero_division=0)

In [3]:
scores=[]; predictions=[]; selections=[]
for fold in range(5):
    tr=np.flatnonzero(folds!=fold); va=np.flatnonzero(folds==fold)
    assert set(groups.iloc[tr]).isdisjoint(groups.iloc[va])
    inner=list(StratifiedGroupKFold(3,shuffle=True,random_state=5612+fold).split(X.iloc[tr],y.iloc[tr],groups.iloc[tr]))
    for a,b in inner:
        assert set(groups.iloc[tr[a]]).isdisjoint(groups.iloc[tr[b]])
        assert set(y.iloc[tr[a]])==set(CLASSES) and set(y.iloc[tr[b]])==set(CLASSES)
    for name,(estimator,grid) in models.items():
        search=GridSearchCV(estimator,grid,scoring=scorer,cv=inner,n_jobs=1,error_score='raise',refit=True)
        search.fit(X.iloc[tr],y.iloc[tr])
        pred=search.predict(X.iloc[va])
        scores.append({'model':name,'fold':fold,'n':len(va),'accuracy':accuracy_score(y.iloc[va],pred),'balanced_accuracy':balanced_accuracy_score(y.iloc[va],pred),'macro_f1':f1_score(y.iloc[va],pred,labels=CLASSES,average='macro',zero_division=0)})
        selections.append({'model':name,'fold':fold,'parameters':search.best_params_,'inner_macro_f1':search.best_score_})
        for i,p in zip(va,pred): predictions.append({'model':name,'fold':fold,'recording_id':df.recording_id.iloc[i],'location_group':groups.iloc[i],'actual':y.iloc[i],'predicted':p})
        print(fold,name,round(scores[-1]['macro_f1'],3),flush=True)
pd.DataFrame(scores).to_csv(OUT/'nested_fold_metrics.csv',index=False)
pd.DataFrame(predictions).to_csv(OUT/'nested_oof_predictions.csv',index=False)
(OUT/'inner_selections.json').write_text(json.dumps(selections,indent=2))
display(pd.DataFrame(scores).groupby('model')[['accuracy','balanced_accuracy','macro_f1']].agg(['mean','std']))
print('No final-test evaluation or replacement of original selected models was performed.')

0 Prior 0.167
0 Multinomial NB 0.499
0 Bernoulli NB 0.531
0 Gaussian NB 0.536
0 Decision tree 0.719
0 Logistic regression 0.414
0 Random forest 0.574
1 Prior 0.177
1 Multinomial NB 0.573
1 Bernoulli NB 0.433
1 Gaussian NB 0.34
1 Decision tree 0.455
1 Logistic regression 0.459
1 Random forest 0.511
2 Prior 0.167
2 Multinomial NB 0.433
2 Bernoulli NB 0.464
2 Gaussian NB 0.315
2 Decision tree 0.384
2 Logistic regression 0.369
2 Random forest 0.442
3 Prior 0.173
3 Multinomial NB 0.488
3 Bernoulli NB 0.548
3 Gaussian NB 0.359
3 Decision tree 0.483
3 Logistic regression 0.506
3 Random forest 0.539
4 Prior 0.167
4 Multinomial NB 0.391
4 Bernoulli NB 0.459
4 Gaussian NB 0.298
4 Decision tree 0.417
4 Logistic regression 0.508
4 Random forest 0.453


accuracy           balanced_accuracy            macro_f1  \
                         mean       std              mean       std      mean   
model                                                                           
Bernoulli NB         0.499134  0.041765          0.490299  0.045194  0.487103   
Decision tree        0.513109  0.123981          0.507519  0.124715  0.491486   
Gaussian NB          0.382941  0.104583          0.380610  0.095441  0.369567   
Logistic regression  0.451571  0.049058          0.447434  0.052754  0.451243   
Multinomial NB       0.483195  0.072992          0.475598  0.069920  0.476910   
Prior                0.342492  0.013008          0.333333  0.000000  0.170041   
Random forest        0.520501  0.049776          0.507529  0.048062  0.503718   

                               
                          std  
model                          
Bernoulli NB         0.049936  
Decision tree        0.132703  
Gaussian NB          0.095910  
Logistic regression  0.060033  
Multinomial NB       0.069093  
Prior                0.004787  
Random forest        0.056235

No final-test evaluation or replacement of original selected models was performed.
